In [11]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import plotly.express as px

import utility_functions as uf

In [130]:
year = 2020
mode = "individual"

mode_dict = {
    "individual": "individual",
    "collaborative": "collab"
}

In [131]:
df_global = (
    pd.read_csv(uf.PATH+"df_stats_subfields_global_yearly.csv")
    .drop_duplicates()
    .query("year == @year")
    .assign(subfield_id_str = lambda df: df.subfield_id.astype(str))
    .set_index("subfield_id_str")
    ["probability_"+mode_dict[mode]]
)

In [132]:
df_prob = (
    pd.read_csv(uf.PATH+"df_prob_yearly.csv")
    .query("year == @year and mode == @mode")
    .drop(["year", "mode"], axis=1)
    .set_index("country")
)

df_im = np.log(
    df_prob
    .div(df_global)
)

In [133]:
# domain_name = "Social Sciences"
# # field_name = "Earth and Planetary Sciences"
# (
#     uf.df_topics
#     [["subfield_id", "subfield_name", "field_name", "domain_name"]]
#     .drop_duplicates()
#     # .query("field_name == @field_name")
#     .query("domain_name == @domain_name")
# )

In [134]:
subfield = "1211"
df_sf_plot = (
    df_im
    .merge(uf.df_country[["alpha-2", "alpha-3"]], left_index=True, right_on="alpha-2", how="left")
    .drop("alpha-2", axis=1)
    .assign(
        subfield_capped=lambda df: df[subfield],
        # subfield_capped=lambda df: np.minimum(df[subfield], df[subfield].quantile(0.75))
    )
)

fig = px.choropleth(
    df_sf_plot,
    locations="alpha-3",
    color="subfield_capped",               # use the categorical version
    locationmode="ISO-3",
    color_discrete_sequence=px.colors.qualitative.Set3,  # discrete color palette
    title=f"{uf.id2subfield_topic[int(subfield)]} ({subfield}) map: {year}, {mode}"
)

fig.update_layout(
    geo=dict(
        showframe=False,
        showcoastlines=True,
        projection_type="natural earth"
    ),
    height=600,
)

fig.show()